In [1]:
import tensorflow as tf
import keras
import numpy as np
import sys
import matplotlib.pyplot as plt

sys.path.append("../")

In [2]:
DATA_DIR = "../mu3e_trigger_data"
MODEL_DIR= "../models"
SIGNAL_PIXEL_FILE = f"{DATA_DIR}/sig_pixel_spacetime.npy"
BACKGROUND_PIXEL_FILE = f"{DATA_DIR}/bg_pixel_spacetime.npy"
SIGNAL_MPPC_FILE = f"{DATA_DIR}/sig_mppc_spacetime.npy"
BACKGROUND_MPPC_FILE = f"{DATA_DIR}/bg_mppc_spacetime.npy"
SIGNAL_ONLY_PIXEL_FILE = f"{DATA_DIR}/sig_only_pixel_spacetime.npy"
SIGNAL_ONLY_MPPC_FILE = f"{DATA_DIR}/sig_only_mppc_spacetime.npy"


bg_pixel_spacetime = np.load(BACKGROUND_PIXEL_FILE)
bg_mppc_spacetime = np.load(BACKGROUND_MPPC_FILE)
sig_pixel_spacetime = np.load(SIGNAL_PIXEL_FILE)
sig_mppc_spacetime = np.load(SIGNAL_MPPC_FILE)


X_pixel = np.concatenate((bg_pixel_spacetime, sig_pixel_spacetime), axis=0)
X_mppc = np.concatenate((bg_mppc_spacetime, sig_mppc_spacetime), axis=0)
y = np.concatenate(
    (np.zeros(bg_pixel_spacetime.shape[0]), np.ones(sig_pixel_spacetime.shape[0])),
    axis=0,
)

# Shuffle the data
indices = np.arange(len(y))
np.random.shuffle(indices)
X_pixel = X_pixel[indices]
X_mppc = X_mppc[indices]
y = y[indices]

input_seq_len = bg_pixel_spacetime.shape[1]
input_dim = bg_pixel_spacetime.shape[2]  # Exclude timestamp

In [3]:
pixel_label = np.zeros(X_pixel.shape[:-1] + (1,))
pixel_label[(X_pixel[:,:,:] == -1).all(axis=-1)] = -1
X_pixel_labelled = np.concatenate(
    (X_pixel, pixel_label), axis=-1
)
mppc_label = np.ones(X_mppc.shape[:-1] + (1,))
mppc_label[(X_mppc[:,:,:] == -1).all(axis=-1)] = -1
X_mppc_labelled = np.concatenate(
    (X_mppc, mppc_label), axis=-1
)


In [4]:
X = np.concatenate(
    (X_pixel_labelled, X_mppc_labelled), axis=1
)